In [1]:
%pylab inline 

import torch 
import torch.nn as nn
import torch.nn.functional as F 
import torchvision
from torchvision import transforms 
from tqdm import trange

Populating the interactive namespace from numpy and matplotlib


In [2]:
# Device Options 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load Data
norm_values = (0.5, 0.5, 0.5) 
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(norm_values,norm_values)
    ])
trainset = torchvision.datasets.CIFAR10(root="datasets/",download=False, train=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root="datasets/", download=False, train=False, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, shuffle=True, batch_size=16)
testloader = torch.utils.data.DataLoader(testset, shuffle=False, batch_size=16)



In [3]:
# Model with basic convnet architecture  
class BasicNet(nn.Module):
    def __init__(self):
        super(BasicNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 12, 5)
        self.pool = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(12, 24, 5)
        self.fc1 = nn.Linear(24*10*10, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = x.view(-1, 24*10*10)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x 

bnet = BasicNet().to(device)


In [4]:
# Write Training function to test on different models 
def trained_model(model, epochs=3, train=False):
    # Loss And Optimization 
    model.train()
    loss_function = nn.CrossEntropyLoss()
    optim = torch.optim.Adam(params=model.parameters())
    
    # Training Loop 
    if train:
        for _ in (t := trange(epochs)):
            for x, y in trainloader:
                x,y = x.to(device), y.to(device)

                # Forward Pass 
                outputs = model(x)
                loss = loss_function(outputs, y)

                # Update Weights 
                optim.zero_grad()
                loss.backward()
                optim.step()
    
    return model
        


      


In [5]:
# Test Model Accuracy on test set
def show_model_accuracy(model):
    model.eval()
    with torch.no_grad():
        for x,y in testloader:
            x,y = x.to(device), y.to(device)
            outputs = model(x)
            n_correct = torch.argmax(outputs, dim=1)
            acc = (n_correct == y).float().mean()*100
        print(f"Model Accuracy = {acc.item():.2f}%")
     





In [7]:
# How good is our basic convnet?
bnet = trained_model(bnet,train=False)
bnet_accuracy = show_model_accuracy(bnet)
bnet_accuracy

Model Accuracy = 75.00%
